# Liver Cirrhosis Survival Prediction
## Notebook 03 — Train · Evaluate · Save Model

**Loads from:** `processed/X_train.csv` · `processed/y_train.csv` · `processed/X_test.csv`  
**Saves to:** `models/model.pkl` · `models/scaler.pkl` · `models/encoders.pkl` · `models/selected_features.pkl`

**What this notebook does:**
1. Mounts Google Drive and sets working directory
2. Loads processed feature sets from Notebook 02
3. Stratified 80/20 train / validation split
4. Scales features with StandardScaler (fit on train only)
5. Applies SMOTE-ENN to training data only to handle class imbalance
6. Trains and tunes three models: Logistic Regression · Random Forest · XGBoost
7. Evaluates each model on the held-out validation set
8. Runs cross-validation on the best model
9. Plots feature importance and confusion matrices
10. Saves best model + scaler + encoders for Flask

---
> **Notebook dependencies:** Requires files in `processed/` saved by Notebook 02.  
> Does **not** require Notebook 02 to be open — just its saved outputs.


### Cell 3.0 — Mount Google Drive & set working directory

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Maria_Sagwa/Cirrhosis_Project')

print("Working directory:", os.getcwd())
print("Files found:", os.listdir())


MessageError: Error: credential propagation was unsuccessful

### Cell 3.1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection   import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing     import StandardScaler
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (accuracy_score, classification_report,
                                       confusion_matrix, roc_auc_score)
from xgboost                   import XGBClassifier
from imblearn.combine          import SMOTEENN

os.makedirs('models',  exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("Libraries loaded ✓")


### Cell 3.2 — Load processed data from Notebook 02

These files were saved by Notebook 02 into the `processed/` folder.  
We load them here directly — no need to rerun preprocessing.

In [ ]:
X      = pd.read_csv('processed/X_train.csv')
y      = pd.read_csv('processed/y_train.csv').squeeze().astype(int)
X_test = pd.read_csv('processed/X_test.csv')

print("=" * 65)
print("3.2  LOADED PROCESSED DATA")
print("=" * 65)
print(f"  X      : {X.shape}   features : {X.columns.tolist()}")
print(f"  y      : {y.shape}   classes  : {sorted(y.unique())}  (0=C, 1=CL, 2=D)")
print(f"  X_test : {X_test.shape}")
print()
print("  y distribution:")
label_names = {0: 'C  — alive', 1: 'CL — transplant', 2: 'D  — deceased'}
for cls, name in label_names.items():
    n = (y == cls).sum()
    print(f"    {name}  {n:5,}  ({n/len(y)*100:.1f}%)")


### Cell 3.3 — Stratified 80/20 train / validation split

**Why stratified?**  
CL (transplant) is only 3.5% of the data — 275 patients.  
A random split could put very few CL cases in the validation set,  
making evaluation unreliable. Stratified split preserves the  
exact class ratio in both halves.

The validation set is kept as **real patient data** throughout —  
SMOTE and scaling are applied only to the training fold.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=" * 65)
print("3.3  TRAIN / VALIDATION SPLIT  (80/20, stratified)")
print("=" * 65)
print(f"  X_train : {X_train.shape}  |  X_val : {X_val.shape}")
print()
print("  y_train distribution:")
for cls, name in label_names.items():
    n = (y_train == cls).sum()
    print(f"    {name}  {n:5,}  ({n/len(y_train)*100:.1f}%)")
print()
print("  y_val distribution:")
for cls, name in label_names.items():
    n = (y_val == cls).sum()
    print(f"    {name}  {n:5,}  ({n/len(y_val)*100:.1f}%)")


### Cell 3.4 — Feature scaling (StandardScaler)

**Why scale?**  
Logistic Regression is sensitive to feature magnitude —  
without scaling, Copper (values ~0–400) would dominate  
Hepatomegaly (values 0 or 1).

**Critical rule:** Fit the scaler **only on training data**,  
then transform val and test. Fitting on all data would leak  
val/test statistics into training.

Tree-based models (Random Forest, XGBoost) are scale-invariant —  
scaling does not affect their predictions, but we scale all  
features so every model runs through the same pipeline.


In [ ]:
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)   # fit + transform on train only
X_val_sc   = scaler.transform(X_val)          # transform only — no fit
X_test_sc  = scaler.transform(X_test)         # transform only — no fit

print("=" * 65)
print("3.4  FEATURE SCALING  (StandardScaler — fit on train only)")
print("=" * 65)
print(f"  X_train mean after scaling : {X_train_sc.mean():.6f}  (should be ≈ 0)")
print(f"  X_train std  after scaling : {X_train_sc.std():.6f}   (should be ≈ 1)")
print()
print("  scaler.pkl will be saved at end of this notebook.")
print("  Flask must use this same scaler to preprocess new patient inputs.")


### Cell 3.5 — SMOTE-ENN (applied to training data only)

**Why SMOTE-ENN?**

| Problem | Solution |
|---------|----------|
| CL class is only 3.5% (275 patients) | SMOTE generates synthetic minority samples |
| Synthetic samples can be noisy | ENN (Edited Nearest Neighbours) removes borderline/noisy samples from all classes |
| Result | Cleaner, more balanced training set with better decision boundaries |

**Critical rule:** SMOTE-ENN is applied **only to X_train_sc / y_train**.  
X_val and X_test are never touched — they must remain real patient data  
for evaluation to be meaningful.


In [ ]:
smote_enn = SMOTEENN(random_state=42)
X_train_bal, y_train_bal = smote_enn.fit_resample(X_train_sc, y_train)

print("=" * 65)
print("3.5  SMOTE-ENN  (training data only)")
print("=" * 65)
print(f"  Before resampling : {X_train_sc.shape[0]:,} rows")
print(f"  After  resampling : {X_train_bal.shape[0]:,} rows")
print()
print("  Balanced y_train distribution:")
unique, counts = np.unique(y_train_bal, return_counts=True)
for cls, n in zip(unique, counts):
    print(f"    {label_names[cls]}  {n:5,}  ({n/len(y_train_bal)*100:.1f}%)")


### Cell 3.6 — Define models

We compare three models:

| Model | Why included |
|-------|-------------|
| **Logistic Regression** | Linear baseline — fast, interpretable, needs scaling |
| **Random Forest** | Ensemble of decision trees — robust, handles non-linearity |
| **XGBoost** | Gradient boosting — typically best on tabular clinical data |

Decision Tree is excluded — Random Forest is strictly better  
(it is an ensemble of decision trees) so a single tree adds no value.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    ),
}

print("=" * 65)
print("3.6  MODELS DEFINED")
print("=" * 65)
for name, model in models.items():
    print(f"  · {name}")


### Cell 3.7 — Hyperparameter tuning (GridSearchCV + StratifiedKFold)

**Why StratifiedKFold (k=5)?**  
Each fold preserves the class ratio — ensures CL (3.5%) appears in every fold.  
A regular KFold could produce a fold with zero CL samples, making AUC unreliable.

**Primary scoring metric: `roc_auc_ovr_weighted`**  
Accuracy is misleading with imbalanced classes — a model that ignores  
CL entirely could still score ~96%. Macro AUC treats all 3 classes fairly.

⚠️ This cell takes several minutes to run — GridSearchCV trains many combinations.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    'Logistic Regression': {
        'C'     : [0.01, 0.1, 1, 10],
        'solver': ['lbfgs', 'saga'],
    },
    'Random Forest': {
        'n_estimators'    : [100, 200],
        'max_depth'       : [None, 10, 20],
        'min_samples_split': [2, 5],
    },
    'XGBoost': {
        'n_estimators' : [200, 300],
        'max_depth'    : [4, 6],
        'learning_rate': [0.05, 0.1],
        'subsample'    : [0.8, 1.0],
    },
}

print("=" * 65)
print("3.7  HYPERPARAMETER TUNING  (GridSearchCV, StratifiedKFold k=5)")
print("=" * 65)
print("  Scoring: roc_auc_ovr_weighted")
print("  This may take a few minutes...\n")

best_models    = {}
tuning_results = {}

for name, model in models.items():
    print(f"  Tuning {name}...")
    grid = GridSearchCV(
        estimator  = model,
        param_grid = param_grids[name],
        cv         = skf,
        scoring    = 'roc_auc_ovr_weighted',
        n_jobs     = -1,
        refit      = True,
        verbose    = 0
    )
    grid.fit(X_train_bal, y_train_bal)
    best_models[name]    = grid.best_estimator_
    tuning_results[name] = {
        'best_params' : grid.best_params_,
        'best_cv_auc' : round(grid.best_score_, 4)
    }
    print(f"    Best params : {grid.best_params_}")
    print(f"    Best CV AUC : {grid.best_score_:.4f}\n")

print("  Tuning complete ✓")


### Cell 3.8 — Evaluate on validation set

We evaluate each tuned model on the **real held-out validation set**  
(X_val_sc — never seen during training or SMOTE).

Metrics per model:
- **Accuracy** — overall correctness
- **Macro AUC-ROC** — primary metric, equal weight to all 3 classes
- **Classification report** — per-class precision, recall, F1
- **Confusion matrix** — where the model confuses classes


In [ ]:
label_name_list = ['C (alive)', 'CL (transplant)', 'D (deceased)']
val_results     = {}

print("=" * 65)
print("3.8  VALIDATION SET PERFORMANCE")
print("=" * 65)

for name, model in best_models.items():
    y_pred = model.predict(X_val_sc)
    y_prob = model.predict_proba(X_val_sc)
    acc    = accuracy_score(y_val, y_pred)
    auc    = roc_auc_score(y_val, y_prob, multi_class='ovr', average='macro')

    val_results[name] = {'accuracy': acc, 'macro_auc': auc}

    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Macro AUC : {auc:.4f}  ← primary metric")
    print(f"\n  Classification Report:")
    print(classification_report(y_val, y_pred,
                                target_names=label_name_list, digits=3))


### Cell 3.9 — Confusion matrices (one per model)

In [ ]:
label_name_list = ['C (alive)', 'CL (transplant)', 'D (deceased)']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confusion Matrices — Validation Set', fontsize=13, fontweight='bold')

for ax, (name, model) in zip(axes, best_models.items()):
    y_pred = model.predict(X_val_sc)
    cm     = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_name_list,
                yticklabels=label_name_list,
                ax=ax, linewidths=0.4)
    auc = val_results[name]['macro_auc']
    ax.set_title(f'{name}\nMacro AUC: {auc:.4f}', fontsize=10)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/03_confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved → outputs/03_confusion_matrices.png")


### Cell 3.10 — Cross-validation on all models

5-fold CV on the balanced training data gives a more stable  
performance estimate than a single train/val split.

**Note:** CV AUC will be higher than Val AUC — this is expected.  
CV runs on SMOTE-balanced (synthetic) data which is easier to classify.  
The Val AUC on real patient data is your honest performance number.

In [ ]:
print("=" * 65)
print("3.10  CROSS-VALIDATION  (StratifiedKFold k=5, balanced train data)")
print("=" * 65)

cv_results = {}
for name, model in best_models.items():
    cv_scores = cross_val_score(
        model, X_train_bal, y_train_bal,
        cv      = skf,
        scoring = 'roc_auc_ovr_weighted',
        n_jobs  = -1
    )
    cv_results[name] = cv_scores
    print(f"  {name:25}  CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print()
print("  Note: CV AUC > Val AUC is expected — CV uses synthetic SMOTE data.")
print("  Trust the Val AUC as your real-world performance estimate.")


### Cell 3.11 — Model comparison summary & best model selection

In [ ]:
print("=" * 65)
print("3.11  MODEL COMPARISON SUMMARY")
print("=" * 65)
print(f"  {'Model':25}  {'Val Accuracy':>13}  {'Val Macro AUC':>14}  {'CV AUC':>10}")
print(f"  {'-'*25}  {'-'*13}  {'-'*14}  {'-'*10}")

for name in val_results:
    acc = val_results[name]['accuracy']
    auc = val_results[name]['macro_auc']
    cv  = cv_results[name].mean()
    star = '  ← best' if name == max(val_results, key=lambda k: val_results[k]['macro_auc']) else ''
    print(f"  {name:25}  {acc:>13.4f}  {auc:>14.4f}  {cv:>10.4f}{star}")

# Select best by Val Macro AUC
best_name  = max(val_results, key=lambda k: val_results[k]['macro_auc'])
best_model = best_models[best_name]

print()
print(f"  ★ Best model : {best_name}")
print(f"    Val Accuracy  : {val_results[best_name]['accuracy']:.4f}")
print(f"    Val Macro AUC : {val_results[best_name]['macro_auc']:.4f}")


### Cell 3.12 — Feature importance (best model)

Feature importance shows which inputs the model relied on most.  
This is clinically important — it validates whether the model  
is making decisions based on medically meaningful signals.

In [ ]:
feature_names = X.columns.tolist()

fig, ax = plt.subplots(figsize=(8, 4))

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    fi_df = (pd.DataFrame({'Feature': feature_names, 'Importance': importances})
             .sort_values('Importance', ascending=True))

    colors = ['#3b82f6' if i == len(fi_df)-1 else '#93c5fd' for i in range(len(fi_df))]
    ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='none')
    ax.set_title(f'Feature Importance — {best_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance score')

    print("=" * 65)
    print(f"3.12  FEATURE IMPORTANCE  ({best_name})")
    print("=" * 65)
    for _, row in fi_df.sort_values('Importance', ascending=False).iterrows():
        bar = '█' * int(row['Importance'] * 100)
        print(f"  {row['Feature']:15}  {row['Importance']:.4f}  {bar}")

elif hasattr(best_model, 'coef_'):
    coef_mean = np.abs(best_model.coef_).mean(axis=0)
    fi_df = (pd.DataFrame({'Feature': feature_names, 'Importance': coef_mean})
             .sort_values('Importance', ascending=True))
    ax.barh(fi_df['Feature'], fi_df['Importance'], color='#93c5fd', edgecolor='none')
    ax.set_title(f'Feature Coefficients — {best_name}', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/03_feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()
print("\nSaved → outputs/03_feature_importance.png")


### Cell 3.13 — Model performance comparison chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')

names    = list(val_results.keys())
accs     = [val_results[n]['accuracy']  for n in names]
aucs     = [val_results[n]['macro_auc'] for n in names]
cv_means = [cv_results[n].mean()        for n in names]
colors   = ['#34d399' if n == best_name else '#93c5fd' for n in names]

# Accuracy
axes[0].bar(names, accs, color=colors, edgecolor='none', width=0.5)
axes[0].set_title('Validation Accuracy', fontsize=11)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Accuracy')
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)

# AUC comparison: Val vs CV
x = np.arange(len(names))
w = 0.3
axes[1].bar(x - w/2, aucs,     width=w, color='#60a5fa', label='Val Macro AUC', edgecolor='none')
axes[1].bar(x + w/2, cv_means, width=w, color='#a78bfa', label='CV AUC (balanced)', edgecolor='none')
axes[1].set_title('Val AUC vs CV AUC', fontsize=11)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('AUC Score')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=15)
axes[1].legend(fontsize=9)
for i, (va, ca) in enumerate(zip(aucs, cv_means)):
    axes[1].text(i - w/2, va + 0.01, f'{va:.3f}', ha='center', fontsize=8)
    axes[1].text(i + w/2, ca + 0.01, f'{ca:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/03_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved → outputs/03_model_comparison.png")


### Cell 3.14 — Save model and all artefacts

We save four things to `models/`:

| File | Used by |
|------|---------|
| `model.pkl` | Flask — loaded at startup to make predictions |
| `scaler.pkl` | Flask — must transform new patient inputs identically to training |
| `encoders.pkl` | Flask — must encode categorical inputs identically to training |
| `selected_features.pkl` | Flask — ensures inputs are in the exact column order the model expects |

**Why save the scaler and encoders separately?**  
The Flask app will receive raw input from the lab technician form.  
It must apply the exact same preprocessing steps in the exact same order  
before passing data to the model. If any step differs, predictions will be wrong.


In [ ]:
# Save best model
joblib.dump(best_model,      'models/model.pkl')

# Save scaler (fitted on training data only)
joblib.dump(scaler,          'models/scaler.pkl')

# Save encoders from Notebook 02
encoders = joblib.load('processed/encoders.pkl')
joblib.dump(encoders,        'models/encoders.pkl')

# Save selected feature list — Flask uses this for column ordering
selected_features = X.columns.tolist()
joblib.dump(selected_features, 'models/selected_features.pkl')

# Save performance summary
perf_df = pd.DataFrame({
    'Model'       : list(val_results.keys()),
    'Val_Accuracy': [v['accuracy']  for v in val_results.values()],
    'Val_MacroAUC': [v['macro_auc'] for v in val_results.values()],
    'CV_AUC_Mean' : [cv_results[k].mean() for k in val_results],
    'Best_Params' : [str(tuning_results[k]['best_params']) for k in val_results],
}).sort_values('Val_MacroAUC', ascending=False)

perf_df.to_csv('outputs/03_model_performance.csv', index=False)

print("=" * 65)
print("3.14  SAVED ARTEFACTS")
print("=" * 65)
print(f"  ✓ models/model.pkl              ← {best_name}")
print(f"  ✓ models/scaler.pkl             ← StandardScaler (fit on train)")
print(f"  ✓ models/encoders.pkl           ← encoding maps for Flask")
print(f"  ✓ models/selected_features.pkl  ← {selected_features}")
print(f"  ✓ outputs/03_model_performance.csv")
print()
print("  Model performance summary:")
print(perf_df[['Model','Val_Accuracy','Val_MacroAUC','CV_AUC_Mean']].to_string(index=False))


### Cell 3.15 — Test single patient prediction

This tests the exact function that Flask will use.  
It takes raw lab values → encodes → scales → predicts → returns result.  
If this cell works correctly, the Flask integration will work too.

In [ ]:
def predict_patient(raw_inputs: dict) -> dict:
    """
    Predict survival outcome for a single new patient.

    Parameters (raw, unscaled — exactly as entered by lab technician):
        raw_inputs: dict with keys matching selected_features
                    e.g. {'Bilirubin': 3.5, 'Prothrombin': 11.2, ...}

    Returns:
        dict with predicted class, probabilities, and confidence
    """
    # Load artefacts
    model_     = joblib.load('models/model.pkl')
    scaler_    = joblib.load('models/scaler.pkl')
    features_  = joblib.load('models/selected_features.pkl')
    encoders_  = joblib.load('models/encoders.pkl')

    # Build DataFrame in exact feature order
    input_df = pd.DataFrame([raw_inputs])[features_]

    # Scale using fitted scaler (transform only, never fit again)
    input_sc = scaler_.transform(input_df)

    # Predict
    pred_class = model_.predict(input_sc)[0]
    pred_proba = model_.predict_proba(input_sc)[0]

    class_map   = {0: 'C', 1: 'CL', 2: 'D'}
    outcome_map = {
        'C' : 'Alive / Censored',
        'CL': 'Liver Transplant',
        'D' : 'Deceased'
    }
    predicted  = class_map[pred_class]
    confidence = float(pred_proba.max())

    return {
        'predicted_status' : predicted,
        'predicted_outcome': outcome_map[predicted],
        'confidence'       : round(confidence, 4),
        'prob_alive'       : round(float(pred_proba[0]), 4),
        'prob_transplant'  : round(float(pred_proba[1]), 4),
        'prob_deceased'    : round(float(pred_proba[2]), 4),
        'confidence_flag'  : 'REVIEW' if confidence < 0.60 else 'OK',
    }


# ── Test with an example patient ────────────────────────────────
example_patient = {
    'Bilirubin'   : 3.5,
    'Prothrombin' : 11.2,
    'Copper'      : 120.0,
    'Hepatomegaly': 1,       # 1 = Yes
    'Stage'       : 3,
    'SGOT'        : 145.0,
    'Spiders'     : 1,       # 1 = Yes
}

result = predict_patient(example_patient)

print("=" * 65)
print("3.15  EXAMPLE SINGLE-PATIENT PREDICTION")
print("=" * 65)
print(f"  Input   : Stage {example_patient['Stage']}, "
      f"Bilirubin={example_patient['Bilirubin']}, "
      f"Copper={example_patient['Copper']}, "
      f"Prothrombin={example_patient['Prothrombin']}")
print()
print(f"  Predicted outcome : {result['predicted_outcome']}")
print(f"  Confidence        : {result['confidence']:.1%}")
print(f"  P(alive)          : {result['prob_alive']:.1%}")
print(f"  P(transplant)     : {result['prob_transplant']:.1%}")
print(f"  P(deceased)       : {result['prob_deceased']:.1%}")
print(f"  Flag              : {result['confidence_flag']}")
